Hands-On #1: Python requests Basics

In [1]:
# Import requests and establish base URL
import requests
url = 'https://jsonplaceholder.typicode.com'

In [2]:
# Exercise 1: First GET Request

response = requests.get(f"{url}/posts/1")

print(response.status_code)

print(response.reason)

200
OK


In [12]:
# Exercise 2: Read Response Metadata

print(response.headers.get("Content-Type", "<missing>"))
print(response.elapsed)
#print(response.elapsed.total_seconds)

# Milliseconds
print(int(response.elapsed.total_seconds() * 1000))

application/json; charset=utf-8
0:00:00.103933
103


In [13]:
# Exercise 3: Parse JSON Safely
try:
    data = response.json()
    userId = data['userId']
    id = data['id']
    title = data['title']
    print(f"userId: {userId}")
    print(f"id: {id}")
    print(f"title: {title}")
except:
    print("JSON error")

userId: 1
id: 1
title: sunt aut facere repellat provident occaecati excepturi optio reprehenderit


In [18]:
# Exercise 4: Send Query Parameters

comments = requests.get(f"{url}/comments", params={"postId": 1}).json()

print(f"Number of comments returned: {len(comments)}")

print(f"Email from first comment: {comments[0]['email']}")

Number of comments returned: 5
Email from first comment: Eliseo@gardner.biz


In [21]:
# Stretch: Handle HTTP and Network Errors

def fetch(url: str):
    try:
        response = requests.get(url, timeout=3)
        response.raise_for_status()
        print('Success with URL:', url)
        print('Response Status Code', response.status_code)
        return response
    except requests.exceptions.Timeout:
        print('Timeout')
    except requests.exceptions.HTTPError as e:
        print(f"An HTTP error occurred: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Request exception: {e}")

# Test cases
print('\nHere is a URL that should load:')
fetch(f"{url}/posts/1")
print('\nHere is a URL that should not exist:')
fetch(f"{url}/not-a-real-route")


Here is a URL that should load:
Success with URL: https://jsonplaceholder.typicode.com/posts/1
Response Status Code 200

Here is a URL that should not exist:
An HTTP error occurred: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/not-a-real-route




Hands-On #2: Authentication with Python requests



In [ ]:
# Exercise 1: Basic Authentication
import requests
from requests.auth import HTTPBasicAuth

url = 'https://httpbin.org'

response = requests.get(f"{url}/basic-auth/student/pass123", auth=HTTPBasicAuth('student', 'pass123'))

print(response.status_code)
print(response.json())

200
{'authenticated': True, 'user': 'student'}


In [26]:
# Exercise 2: Bearer Token Header

response = requests.get(f"{url}/bearer", headers={"Authorization": "Bearer abc123"})

print(response.status_code)
print(response.json())

200
{'authenticated': True, 'token': 'abc123'}


In [29]:
# Exercise 3: API Key in Header and Query String

response1 = requests.get(f"{url}/get", headers={"X-API-Key": "demo-key-001"})
response2 = requests.get(f"{url}/get", params={"api_key": "demo-key-001"})

print(response1.status_code)
print(response1.json())
print()
print(response2.status_code)
print(response2.json())

200
{'args': {}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate, zstd', 'Host': 'httpbin.org', 'User-Agent': 'python-requests/2.34.2', 'X-Amzn-Trace-Id': 'Root=1-6a70c952-4b7af7a60453c949295ca608', 'X-Api-Key': 'demo-key-001'}, 'origin': '128.2.149.6', 'url': 'https://httpbin.org/get'}

200
{'args': {'api_key': 'demo-key-001'}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate, zstd', 'Host': 'httpbin.org', 'User-Agent': 'python-requests/2.34.2', 'X-Amzn-Trace-Id': 'Root=1-6a70c952-14e57f7b2186f0e52e31470d'}, 'origin': '128.2.149.6', 'url': 'https://httpbin.org/get?api_key=demo-key-001'}


In [30]:
# Exercise 4: Session Cookies

session = requests.Session()
session.get(f"{url}/cookies/set/course_token/python-lesson-10")
response = session.get(f"{url}/cookies")

print(response.status_code)
print(response.json())

200
{'cookies': {'course_token': 'python-lesson-10'}}


In [62]:
# Problem 4 ALT
address = url

print('Checking Cookies')
with requests.Session() as session:
    response = session.get(f"{address}/cookies/set/course_token/python-lesson-10")
    print(response.text)
    response = session.get(f"{address}/cookies/set/course_token/my-second-cookie")
    response = session.get(f"{address}/cookies")
    print(response.request.headers)
print()

Checking Cookies
{
  "cookies": {
    "course_token": "python-lesson-10"
  }
}

{'User-Agent': 'python-requests/2.34.2', 'Accept-Encoding': 'gzip, deflate, zstd', 'Accept': '*/*', 'Connection': 'keep-alive', 'Cookie': 'course_token=my-second-cookie'}



In [60]:
# Stretch: Auth Wrapper with Safe Secret Handling
import os

def auth_get(url, token_env="API_TOKEN"):
    if not token_env:
        print('Missing token.')
        return None

    try:
        response = requests.get(url, headers={"Authorization": f"Bearer {os.getenv(token_env)}"}, timeout=3)
        response.raise_for_status()
        print('Success!')
        print(response.status_code)
        return response

    except requests.exceptions.Timeout:
        print('Timeout')
    except requests.exceptions.HTTPError as e:
        print(f"An HTTP error occurred: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Request exception: {e}")

    return None


# Without a token, will raise exception
auth_get("https://httpbin.org/bearer", token_env=None)

# Set env var
API_TOKEN = "Authorization: Bearer 12345"
auth_get("https://httpbin.org/bearer", API_TOKEN)

Missing token.
Success!
200


<Response [200]>